# 🏗️ Project Starter — Tokenizer & Mini-LM dari Nol (Bab 9)

> **Prasyarat:** semua test di `tests/` HIJAU (`python -m unittest discover -s tests -v`).
> Notebook ini adalah tempat **eksperimen & analisis** — bagian penilaian selain test.

Kerjakan berurutan. Setiap eksperimen diakhiri **Pertanyaan** — jawab dengan kata-kata
sendiri + merujuk angka yang kamu lihat. Isi juga `RUBRIK.md` setelah selesai.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from tok_mini.bpe import apply_merges, ke_simbol, latih_bpe, normalisasi
from tok_mini.data import CORPUS_TEKS, LM_D, LM_SEQ, LM_VOCAB, SAMPLE_LOGITS, SEED
from tok_mini.embedding import analogi, cari_mirip, cos_sim, latih_embedding
from tok_mini.lm import init_params, prediksi_berikut, train_lm
from tok_mini.position import positional_encoding
from tok_mini.sampling import sample_greedy, sample_topp, sample_topk
from tok_mini.tokenizer import buat_vocab, decode_teks, encode_teks

CORPUS = normalisasi(CORPUS_TEKS)
print('corpus:', CORPUS)
print('SEED =', SEED, '(terkunci — eksperimenmu harus reproducible)')

## Eksperimen 1 — Anatomi Token

Encode teks yang mengandung kata dari corpus DAN kata baru (tapi satu alfabet).
Lihat kata mana yang menjadi sedikit token dan mana yang pecah berantakan.

In [ ]:
merges, _ = latih_bpe(CORPUS, 4)
vocab = buat_vocab(CORPUS, 4)

teks = "newer lowliest newer newest wider low lowlier"
hasil = encode_teks(teks, merges, vocab)
for kata, toks in zip(normalisasi(teks), hasil):
    print(f'{kata:<9} ({len(toks)} token) -> {list(toks)}')

total_token = sum(len(t) for t in hasil)
total_kata = len(hasil)
print(f'\nratio token/kata: {total_token / total_kata:.2f}')

**Pertanyaan 1.** Kata mana yang paling hemat token, dan kenapa (lihat frekuensi di
corpus)? Kata baru seperti `lowliest` dipecah jadi apa — dan kenapa TIDAK muncul
error? Kaitkan dengan cara API LLM menghitung biaya.

## Eksperimen 2 — Trade-off Jumlah Merge

Vocabulary kecil = tabel kecil tapi teks panjang; vocabulary besar sebaliknya.
Ukur trade-off-nya.

In [ ]:
uji = "newer lowliest newer newest wider"
print(f'{"n_merge":>7} | {"vocab":>5} | {"total token":>11} | merge ke-…')
print('-' * 60)
for n in (0, 1, 2, 4, 8):
    m, _ = latih_bpe(CORPUS, n)
    v = buat_vocab(CORPUS, n)
    tot = sum(len(t) for t in encode_teks(uji, m, v))
    ringkas = ', '.join(f'{p[0]}+{p[1]}' for p, _ in m[-2:]) if m else '-'
    print(f'{n:>7} | {len(v):>5} | {tot:>11} | {ringkas}')

**Pertanyaan 2.** Bagaimana bentuk kurva total token terhadap n_merge — turun linear
atau melandai? Merge mana yang memberi penghematan terbesar per token? Kenapa
tokenzier produksi berhenti di ~50rb–250rb merge, tidak 1 juta?

## Eksperimen 3 — Struktur di Ruang Embedding

Embedding hasil `latih_embedding` menyimpan struktur kelompok. Lihat matrix
cosine-nya, lalu uji retrieval & analogi.

In [ ]:
W = latih_embedding(idx_vokal=(0, 4, 6), vocab=8, d=8, epochs=200, seed=SEED)
C = np.array([[cos_sim(W[i], W[j]) for j in range(8)] for i in range(8)])
print('matrix cosine (baris/kolom = token 0..7; kelompok vokal = 0, 4, 6):')
print(np.round(C, 2))

print('\ncari_mirip(token 0):', cari_mirip(W, W[0], exclude={0}, top_n=3))
print('analogi(1, 0, 2): token', analogi(1, 0, 2, W), ' — konsonan->vokal?')

plt.figure(figsize=(5, 4))
plt.imshow(C, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='cosine')
plt.title('Struktur kelompok di matrix cosine')
plt.tight_layout(); plt.show()

**Pertanyaan 3.** Blok mana di matrix cosine yang terang, dan kenapa muncul padahal
tidak ada satupun label diberikan saat training? Lalu naikkan `d` ke 64 dan hitung
mean & std cosine dari 60 pasangan vektor ACAK (numpy) pada d=8 vs d=64 — pola apa
yang muncul, dan apa implikasinya untuk retrieval berbasis cosine (Bab 12)?

## Eksperimen 4 — Attention: Encoder vs Decoder

Satu set Q/K/V, dua mode. Bandingkan pattern attention-nya sebagai heatmap.

In [ ]:
from tok_mini.attention import self_attention

rng = np.random.default_rng(SEED)
Q = rng.normal(0, 1, (6, 8)); K = rng.normal(0, 1, (6, 8)); V = rng.normal(0, 1, (6, 8))

out_enc, attn_enc = self_attention(Q, K, V, causal=False)
out_dec, attn_dec = self_attention(Q, K, V, causal=True)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(attn_enc, cmap='viridis', vmin=0, vmax=1)
axes[0].set_title('Encoder (lihat semua)')
axes[1].imshow(attn_dec, cmap='viridis', vmin=0, vmax=1)
axes[1].set_title('Decoder (causal mask)')
for ax in axes:
    ax.set_xlabel('token dilihat'); ax.set_ylabel('token pemilik attention')
plt.tight_layout(); plt.show()

print('baris terakhir identik?', np.allclose(attn_enc[-1], attn_dec[-1]))
print('baris pertama identik?', np.allclose(attn_enc[0], attn_dec[0]))

**Pertanyaan 4.** Baris mana yang identik antara kedua mode, dan kenapa BARU baris
terakhir yang berbeda jauh? Kalau causal mask dihapus saat training GPT tapi tetap
dipakai saat inference, apa gejalanya?

## Eksperimen 5 — Distribusi Sampling

Logits yang sama, tiga strategi. Histogram hasil sampling menjelaskan 'kepribadian'
model.

In [ ]:
logits = np.asarray(SAMPLE_LOGITS)
rng = np.random.default_rng(SEED)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for ax, (nama, sampler) in zip(axes, [
        ('greedy (deterministik)', lambda: sample_greedy(logits)),
        ('top-k=4, T=1.0', lambda: sample_topk(logits, 4, rng)),
        ('top-p=0.6, T=1.0', lambda: sample_topp(logits, 0.6, rng))]):
    sampel = [sampler() for _ in range(300)]
    ax.hist(sampel, bins=np.arange(-0.5, 8.5), rwidth=0.8)
    ax.set_title(nama); ax.set_xlabel('token')
axes[0].set_ylabel('frekuensi (300 sample)')
plt.tight_layout(); plt.show()

rng2 = np.random.default_rng(0)
for T in (0.5, 1.0, 2.0):
    ids = [sample_topk(logits, 4, rng2, T=T) for _ in range(1000)]
    print(f'T={T:>4}: token unik = {sorted(set(ids))}')

**Pertanyaan 5.** Mengapa histogram greedy hanya satu batang, sementara top-p=0.6
punya tiga? Kalau kamu membangun chatbot layanan pelanggan, pilih T dan top-p berapa
— dan apa risiko pilihanmu?

## Eksperimen 6 — Generalisasi Mini-LM

Model dilatih HANYA pada `[0,1,0,1,0,1,0,1]`. Sekarang uji pola yang tidak pernah
dilihatnya — mulai dari token 1, atau lebih panjang.

In [ ]:
trained, losses = train_lm(init_params(SEED, vocab=LM_VOCAB, d=LM_D), LM_SEQ,
                           epochs=120, lr=0.05)
print(f'loss: {losses[0]:.3f} -> {losses[-1]:.4f} (baseline buta ln(4) = 1.386)')

uji_pola = [[0], [0, 1], [0, 1, 0], [1], [1, 0], [1, 0, 1], [0, 1, 0, 1]]
for pola in uji_pola:
    print(f'{pola} -> prediksi token {prediksi_berikut(trained, pola)}')

plt.figure(figsize=(7, 3.2))
plt.plot(losses)
plt.axhline(np.log(LM_VOCAB), color='gray', ls='--', lw=1, label='baseline buta ln(4)')
plt.xlabel('epoch'); plt.ylabel('next-token CE'); plt.legend()
plt.title('Mini-LM training'); plt.tight_layout(); plt.show()

**Pertanyaan 6.** Model memprediksi benar untuk prompt yang DIMULAI dari token 1
(`[1] -> 0`) padahal training selalu mulai dari 0 — bahkan pola 4-token `[0,1,0,1]`
pun diprediksi benar. Apa yang sebenarnya dipelajari model — urutan absolut atau
hubungan bergantian? Kaitkan dengan mengapa LLM besar bisa melanjutkan teks yang
tidak pernah ada di data training.

## 📋 Pertanyaan Analisis (bagian penilaian!)

Jawab 6 pertanyaan di atas dalam `RUBRIK.md` (atau notebook ini). Standar: kata-kata
sendiri + merujuk angka eksperimenmu. Jawaban "kayaknya" = nilai minimum.

1. Anatomi token: kata sering vs kata baru, dan implikasi biaya API.
2. Trade-off jumlah merge: bentuk kurva & kenapa tidak merge sebanyak mungkin.
3. Struktur embedding tanpa label; efek menaikkan dimensi.
4. Simetri encoder/decoder; gejala mask hilang saat training.
5. Kepribadian tiap strategi sampling; konfigurasi untuk chatbot.
6. Generalisasi pola bergantian — hubungan vs hafalan.

Setelah semua terjawab dan RUBRIK terisi: bandingkan dengan `solusi.ipynb` dan
`solusi/tok_mini_ref.py`.